# Cut LLM Costs 80% With Semantic Caching in Agent Command Center

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/falcon-ai-page/command-center/semantic-caching.ipynb)
[![View on GitHub](https://img.shields.io/badge/View_on_GitHub-181717?logo=github&logoColor=white)](https://github.com/future-agi/cookbooks/blob/cookbook/falcon-ai-page/command-center/semantic-caching.ipynb)

| Time | Difficulty |
|------|------------|
| 10 min | Beginner |

By the end of this cookbook you will have exact + semantic caching enabled at the Agent Command Center gateway, paraphrased duplicate prompts returning cached answers in sub-100ms with near-zero cost, and a way to bypass or invalidate the cache when you need fresh answers. The only application-code change is pointing your OpenAI SDK at the gateway base URL.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- Agent Command Center API key starting with `sk-agentcc-` (Settings → API Keys)
- At least one LLM provider configured in [Agent Command Center → Providers](https://docs.futureagi.com/docs/command-center/features/providers)
- Python 3.9+


## Install

Install the OpenAI and Agent Command Center SDKs and set your API key.

In [ ]:
%pip install openai agentcc

In [ ]:
import os
os.environ["AGENTCC_API_KEY"] = "sk-agentcc-your-key"


## What is Agent Command Center?

Agent Command Center is a model gateway in front of your LLM provider calls. Point your OpenAI SDK at the gateway base URL (`https://gateway.futureagi.com/v1`) and every request gets routed through it, where the platform applies caching, routing, fallback, cost tracking, and observability before forwarding to the underlying provider.

This cookbook turns on the **caching** layer specifically. The gateway has two cache tiers: an L1 **exact** cache that matches byte-identical prompts, and an L2 **semantic** cache that matches paraphrased prompts by meaning. The five steps below send a baseline request, enable exact caching from the dashboard, turn on the semantic L2 layer, measure the savings on a realistic batch, and show how to bypass or invalidate the cache when you need fresh answers.


## Step 1: Send a baseline request and note the cost

Before turning anything on, send one request through the gateway with caching disabled. This is your "no caching" control. Every gateway response carries `x-agentcc-cache`, `x-agentcc-cost`, and `x-agentcc-latency-ms` headers, so you can read the exact cost and latency you're paying right now and compare it to the cached numbers in the steps that follow.

In [ ]:
from openai import OpenAI

API_KEY = os.environ["AGENTCC_API_KEY"]

client = OpenAI(
    api_key=API_KEY,
    base_url="https://gateway.futureagi.com/v1",
)

r = client.chat.completions.with_raw_response.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is your return policy?"}],
)
print(f"cache:   {r.headers.get('x-agentcc-cache')}")
print(f"cost:    ${r.headers.get('x-agentcc-cost')}")
print(f"latency: {r.headers.get('x-agentcc-latency-ms')}ms")

`x-agentcc-cache` is empty or `miss` on a fresh call. The cost and latency are what you'd pay every time without caching.

## Step 2: Turn on exact caching in the dashboard

The first cache layer is **L1 exact match**. The gateway hashes the exact request (model, messages, temperature, all parameters) and stores the response under that hash. Any future request with the same hash returns the cached response without ever calling the provider. It's the cheapest, fastest layer to enable, and it kicks in automatically the moment caching is on.

In the dashboard, go to **Gateway → Providers → Cache** and click **Configure Cache**. Toggle:

- **Enable Response Cache**: on
- **Default TTL**: `1h` (cached entries expire after an hour. Set this based on how stale your data can be before you need a fresh model call.)

Save. Caching is now active for every request through the gateway. The Cache Configuration card shows `Enabled: Yes` with `L1 Backend: memory` (the L1 layer is always exact-match; `memory` just means it's stored in-process. You can switch to Redis or disk for a multi-instance gateway). `Semantic Cache: Disabled` confirms only exact matches are served right now. No client change required. Run the same prompt twice:

In [ ]:
prompt = [{"role": "user", "content": "What is your return policy?"}]

r1 = client.chat.completions.with_raw_response.create(model="gpt-4o-mini", messages=prompt)
print(f"call 1: {r1.headers.get('x-agentcc-cache')} | ${r1.headers.get('x-agentcc-cost')}")

r2 = client.chat.completions.with_raw_response.create(model="gpt-4o-mini", messages=prompt)
print(f"call 2: {r2.headers.get('x-agentcc-cache')} | ${r2.headers.get('x-agentcc-cost')}")

Call 1 is `miss`. Call 2 is `hit_exact`, instant, with `$0` provider cost. Exact caching is fast and free, but only helps when prompts are byte-identical.

> **Tip.** Use cache **namespaces** to isolate environments or experiments. Set `x-agentcc-cache-namespace: staging` on a request to keep its cache separate from production. Each namespace is independent. A `prod` hit won't leak into `staging`.

## Step 3: Switch to semantic caching for paraphrased prompts

Real users don't ask the same question the same way twice. *"What is your return policy?"* and *"Can I return a product?"* are the same question to a human but byte-different to the L1 hash, so L1 alone misses both. The **L2 semantic cache** fixes that. The gateway embeds each prompt into a vector, looks for a previously-cached vector within a similarity threshold, and returns that response. L2 only runs after L1 misses, so byte-identical prompts still take the fast path.

In the same **Configure Cache** dialog, enable:

- **L2 Semantic Cache**: on
- **Threshold**: `0.92` (similarity, 0 to 1, higher is stricter. 0.92 catches paraphrases without colliding unrelated questions.)

The same client code now matches paraphrases:

In [ ]:
prompts = [
    "What is your return policy?",
    "Can I return a product I bought?",
    "How do refunds work at your store?",
    "Tell me about returning items.",
]
for p in prompts:
    r = client.chat.completions.with_raw_response.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": p}],
    )
    print(f"{(r.headers.get('x-agentcc-cache') or 'miss'):14} | ${r.headers.get('x-agentcc-cost')} | {p}")

The first prompt is `miss`; the rest are paraphrases above the 0.92 similarity threshold and come back as `hit_semantic` with near-zero cost.

> **Tip.** Tune the threshold carefully. Too low (e.g., 0.7) and unrelated questions collide; too high (e.g., 0.99) and you only catch near-exact matches. Start at 0.92 and adjust based on your hit rate vs false-positive rate.

## Step 4: Measure the savings

Step 2 and 3 each verified a single hit. To see what production traffic actually saves, run a realistic batch where each unique question repeats several times (mimicking what your support bot or FAQ assistant gets all day). The first occurrence of each question pays the provider; every repeat hits cache for near-zero cost and sub-second latency. Tally the breakdown across the batch and you've measured your actual hit rate and dollar savings.

In [ ]:
import time
from collections import Counter

batch = [
    "What is your return policy?",
    "Can I return a product?",
    "How do I get a refund?",
    "What's the shipping cost?",
    "How long does shipping take?",
    "Do you ship internationally?",
] * 5  # 30 calls total

tally = Counter()
total_cost = 0.0
start = time.time()

for p in batch:
    r = client.chat.completions.with_raw_response.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": p}],
    )
    tally[r.headers.get("x-agentcc-cache") or "miss"] += 1
    total_cost += float(r.headers.get("x-agentcc-cost") or 0)

print(f"cache results: {tally}")
print(f"total cost:    ${total_cost:.5f}")
print(f"wall time:     {time.time() - start:.1f}s")

Expect ~80% hits after the first pass over each unique question (`hit_exact` for byte-identical, `hit_semantic` for paraphrases). Compare the total cost against the same batch with caching disabled. That's your savings.

## Step 5: Bypass or invalidate the cache when you need fresh answers

Caching only helps if you can defeat it when you need to. Two real situations: you're testing a prompt change and need to see the new model output, or you've shipped a fix and want to invalidate every cached entry that was generated under the old prompt. The gateway gives you both.

For one-off bypass, send `x-agentcc-cache-force-refresh: true` on a single request. The gateway skips the cache read but still writes the new response back, so subsequent identical calls hit the refreshed entry.

In [ ]:
r = client.chat.completions.with_raw_response.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is your return policy?"}],
    extra_headers={"x-agentcc-cache-force-refresh": "true"},
)
print(f"forced refresh: {r.headers.get('x-agentcc-cache')}")  # miss, then re-cached

For a global wipe after a prompt-template update, route your traffic to a fresh namespace by setting `x-agentcc-cache-namespace: support-v2` instead of `support`. The old cache stays available to anything still pointed at `support`.

## What you solved

Repetitive user questions (the kind any production support bot, FAQ assistant, or knowledge-base agent sees daily) now return cached answers for paraphrased duplicates instead of paying the provider for the same call twice. Sub-100ms responses for cache hits, full cost only for the first unique question, and a single `x-agentcc-cache` header on every response so you can audit hit rates from production logs.

> **Check.** You enabled exact then semantic caching in the dashboard, watched paraphrased prompts return cached responses with `x-agentcc-cache: hit_semantic`, and measured the cost drop on a realistic batch, without changing application code beyond pointing at the gateway.

- **Pay-for-every-call cost** (no caching at all): solved by enabling the L1 exact cache in the dashboard with one toggle
- **Cache misses on paraphrased duplicates** (exact-only caching): solved by turning on the L2 semantic cache with a similarity threshold
- **No visibility into cache hit rate**: solved by the `x-agentcc-cache`, `x-agentcc-cost`, `x-agentcc-latency-ms` response headers on every request
- **Stale cached responses after a prompt change**: solved by `x-agentcc-cache-force-refresh: true` per request, or by routing to a fresh `x-agentcc-cache-namespace`

## Explore further

- **[Caching](https://docs.futureagi.com/docs/command-center/features/caching)**: Cache modes, TTL, invalidation, and per-org configuration
- **[Routing & Reliability](https://docs.futureagi.com/docs/command-center/features/routing)**: Weighted routing, fallback, and cost-optimized strategies
- **[Cost tracking](https://docs.futureagi.com/docs/command-center/features/cost-tracking)**: Per-request cost reporting and budget alerts